In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("fraud_detection.silver.transactions_enriched")

country_risk_profile = (
    silver_df
    .filter(F.col("country_name").isNotNull())
    .groupBy("country_name", "country_iso_code")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("is_fraudulent").alias("fraud_count"),
        F.round(F.sum("is_fraudulent") / F.count("*") * 100, 2).alias("fraud_rate_pct")
    )
    .filter(F.col("total_transactions") >= 100)  # exclude tiny-sample countries where a rate is statistical noise
    .orderBy(F.desc("fraud_rate_pct"))
)

display(country_risk_profile.limit(20))
display(country_risk_profile.orderBy("fraud_rate_pct").limit(10))

print(f"Total countries in profile (>=100 transactions): {country_risk_profile.count()}")

In [0]:
from pyspark.sql import functions as F
import math

overall_fraud_rate = silver_df.agg(F.avg("is_fraudulent")).collect()[0][0]
print(f"Overall baseline fraud rate: {overall_fraud_rate:.4f}")

country_risk_with_significance = country_risk_profile.withColumn(
    "expected_fraud_count", F.col("total_transactions") * F.lit(overall_fraud_rate)
).withColumn(
    "std_error_pct", F.sqrt(F.lit(overall_fraud_rate) * (1 - F.lit(overall_fraud_rate)) / F.col("total_transactions")) * 100
).withColumn(
    "deviation_in_std_errors",
    (F.col("fraud_rate_pct") - F.lit(overall_fraud_rate * 100)) / F.col("std_error_pct")
)

display(
    country_risk_with_significance
    .orderBy(F.desc(F.abs("deviation_in_std_errors")))
    .select("country_name", "total_transactions", "fraud_rate_pct", "std_error_pct", "deviation_in_std_errors")
    .limit(20)
)

In [0]:
country_risk_with_significance.write.mode("overwrite").saveAsTable("fraud_detection.gold.country_risk_profile")